# Извличане на изречения Науатл-Испански

## 1. Описание на задачата

Вашата задача е да работите с лингвистичен изследователски екип, който изучава науатл - местен език, говорен в Мексико от над 1.5 милиона души. Екипът е събрал паралелен корпус от двойки изречения на науатл и испански, но се нуждае от ефективна система за намиране на съответстващи преводи в големи двуезични колекции.

Вашата задача е да изградите **система за междуезиково представяне (cross-language)**: дадено е изречение на науатл, целта е да се намери правилния му испански превод измежду набор от 10 кандидата.

Това е фундаментален проблем в обработката на естествен език с приложения в:
- Оценка на качеството на машинен превод
- Извличане на паралелен корпус от уеб данни
- Междуезиково представяне на информация

### За езика науатл

Науатл е **аглутинативен език**, което означава, че думите се образуват чрез свързване на морфеми (малки смислови единици). Това е много различно от испанския или английския, където думите са по-изолирани.

**Примери за морфология на науатл:**

| Науатл | Разбивка | Значение |
|--------|----------|----------|
| `nocal` | `no-cal` = моя-къща | "моята къща" |
| `nimitztlazohtla` | `ni-mitz-tlazohtla` = аз-теб-обичам | "обичам те" |
| `tlacualli` | `tla-cua-lli` = нещо-ям-нещо | "храна" |
| `cihuaconetl` | `cihua-conetl` = жена-дете | "момиче" |
| `tlenamacac` | `tle-nama-cac` = огън-продавам-човек | "продавач на огън" |

**Защо това е важно за NLP:**
- Стандартните токенизатори (проектирани за испански/английски) може да не разделят правилно думите на науатл
- Една дума на науатл може да съответства на цяла фраза на испански
- Моделите на морфемите могат да помогнат за идентифициране на преводи (напр. споделени корени, заемки)

## 2. Данни

### Структура на данните

**Тренировъчен набор**: **10 000 паралелни двойки изречения**
- Формат: CSV с колони `nahuatl, spanish`
- Всеки ред е верифицирана двойка превод
- Използвайте го за изучаване на междуезиково семантично сходство

**Валидационен набор**: **500 заявки за извличане**
- Използвайте го за проверка на вашето решение
- Всяка заявка съдържа:
  - 1 изречение на науатл
  - 10 испански кандидат-изречения (означени `spanish_0` до `spanish_9`)
  - Точно 1 кандидат е правилният превод (предоставена е истинска стойност)

**Тестов набор**: **500 заявки за извличане**
- Същата структура като валидационния набор

## 3. Задача

За всяка заявка в тестовия набор:
- **Вход**: 1 изречение на науатл + 10 испански кандидат-изречения
- **Изход**: Едно цяло число (0-9), указващо кой испански кандидат е правилният превод

### Формат на предаване

Генерирайте `Task_2_USER_ID_submission.csv`, съдържащ:

- 500 реда (по един за всяка тестова заявка)
  - Всеки ред: едно цяло число 0-9
  - Без заглавен ред
  - Сменете `USER_ID` с вашия идентификационен номер. 

## 4. Оценяване

**Основна метрика**: **Accuracy@1** (Точност на първи опит)

```
Accuracy@1 = (Брой заявки с правилна предсказана позиция) / (Общ брой заявки)
```
Класирането ще се определи на базата на Accuracy@1 върху **скрит тестов набор** (held-out test set). Предоставеният валидационен набор (с истински стойности) е за локална оценка на вашето решение — използвайте го, за да проверявате и подобрявате подхода си преди предаване.

## 5. Базов подход

Тази тетрадка имплементира прост базов подход, използвайки предварително обучени многоезични вграждания (semantic embeddings):
- Модел: `paraphrase-multilingual-MiniLM-L12-v2`
- Метод: Кодиране на заявката на науатл и всички испански кандидати, избор на кандидата с най-високо косинусово сходство
- Очакван резултат: около **33%**

### Как да подобрите резултата?

Към този проблем може да се подходи по много разнообразни начини: съставяне на ръчни признаци (features) които да се възползват от морфологията на езика, fine-tune на модели за семантични сравнения или ансамблиране на няколко такива, вторични модели за пре-ранкиране на резултата. 

Експериментирайте смело и открийте какво работи най-добре!

## Предаване на решението:
Заменете `USER_ID` със вашият индентификационен номер.
* `Task_2_USER_ID_submission.csv` с предсказанията от най-добрият ви модел
* изпълнената jupyter notebook наименована като `Task_2_USER_ID.ipynb`

## Настройка

In [2]:
import pandas as pd
import numpy as np
from typing import List

## Зареждане на данни

Зареждане на тренировъчния, валидационния и тестовия набор от предоставените CSV файлове.

In [4]:
# Зареждане на данните
DATA_DIR = "."

print("Зареждане на данни...")
train_df = pd.read_csv(f"{DATA_DIR}/training_set.csv")
val_df = pd.read_csv(f"{DATA_DIR}/validation_set.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test_set.csv")
val_ground_truth = pd.read_csv(f"{DATA_DIR}/ground_truth_validation.csv")['correct_position'].tolist()

print(f"Тренировъчен набор: {len(train_df)} двойки")
print(f"Валидационен набор: {len(val_df)} заявки (с истински стойности за локална оценка)")
print(f"Тестов набор: {len(test_df)} заявки")

Зареждане на данни...
Тренировъчен набор: 10000 двойки
Валидационен набор: 500 заявки (с истински стойности за локална оценка)
Тестов набор: 500 заявки


In [5]:
# Преглед на тренировъчния набор
print("Преглед на тренировъчния набор:")
print(train_df.head())
print(f"\nКолони: {list(train_df.columns)}")

Преглед на тренировъчния набор:
                                             nahuatl  \
0              In mochiua kampa eski komo se kitoka.   
1  Axcan sábado yn ic 22 mani metztli junio de 16...   
2  Yn ipan axcan lunes a veynte dias del mes de a...   
3  Yn ocontlallique niman concuique yn imitac, au...   
4  Cualica, nehua nis naquiz notlaquen tlitic, ic...   

                                             spanish  
0        Se da en cualquier parte si uno lo siembra.  
1  El sábado 22 de junio de 1613 se puso un nuevo...  
2  Oy lunes a veynte dias del mes de abril de mil...  
3  Después de colocarlo allí, los aztecas sacaron...  
4  Bueno, yo voy a ponerme mi traje negro, con el...  

Колони: ['nahuatl', 'spanish']


In [6]:
# Преглед на структурата на валидационния набор
print("Преглед на валидационния набор:")
print(f"Колони: {list(val_df.columns)}")
print(f"\nПримерна заявка:")
print(f"  Науатл: {val_df.iloc[0]['nahuatl'][:80]}...")
print(f"\n  Кандидати:")
for i in range(10):
    print(f"    spanish_{i}: {val_df.iloc[0][f'spanish_{i}'][:60]}...")

Преглед на валидационния набор:
Колони: ['nahuatl', 'spanish_0', 'spanish_1', 'spanish_2', 'spanish_3', 'spanish_4', 'spanish_5', 'spanish_6', 'spanish_7', 'spanish_8', 'spanish_9']

Примерна заявка:
  Науатл: Pero niquintitlani in tocnihuan para ma ca yej xijqui onquisas on tlen cuajli yo...

  Кандидати:
    spanish_0: Aquí está Moquihuitzin, que combatió a tlaxcaltecas, huexotz...
    spanish_1: El domingo 3 de mayo de 1609, en la ciudad de los Ángeles Cu...
    spanish_2: Tu hermana es hermosa....
    spanish_3: Y por decirlo así, en la persona de Abraham también Leví, el...
    spanish_4: a pesar de los falsos hermanos quienes se infiltraron secret...
    spanish_5: Su fama corrió por toda Siria, y le trajeron todos los que t...
    spanish_6: Si decimos que no tenemos pecado, nos engañamos a nosotros m...
    spanish_7: Entonces respondió Jesús y le dijo: --¡Oh mujer, grande es t...
    spanish_8: Pero he enviado a estos hermanos para que el orgullo que ten...
    spanish_9: Ento

---

# ВАШЕТО РЕШЕНИЕ

**Редактирайте тази секция, за да имплементирате вашия подход.**

Имате достъп до:
- `train_df`: 10 000 тренировъчни двойки с колони `['nahuatl', 'spanish']`
- `val_df`: 500 валидационни заявки с колони `['nahuatl', 'spanish_0', ..., 'spanish_9']`
- `test_df`: 500 тестови заявки (същата структура като val_df)

**Изискване:** Имплементирайте функцията `predict` по-долу. Тя ще бъде извикана веднъж за всяка заявка.

```python
def predict(nahuatl: str, candidates: List[str]) -> int:
    """
    Аргументи:
        nahuatl: Изречението на науатл за превод
        candidates: Списък от 10 испански кандидат-изречения
    
    Връща:
        Индекс (0-9) на предсказания правилен превод
    """
```

Можете да добавите всякакъв код над функцията `predict` (зареждане на модели, обучение, помощни функции и др.).

In [ ]:
# ============================================================================
# ВАШЕТО РЕШЕНИЕ - НАЧАЛО
# ============================================================================
# Добавете вашите импорти, зареждане на модели, код за обучение и помощни
# функции тук. Единственото изискване е да имплементирате функцията predict().
# ============================================================================
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer, util
import torch

# Зареждане на предварително обучен многоезичен модел
print("Зареждане на модел...")
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')
print("Моделът е зареден!")

Зареждане на модел...
Моделът е зареден!


In [8]:
def predict(nahuatl: str, candidates: List[str]) -> int:
    """
    Предсказва кой испански кандидат е правилният превод.
    
    Аргументи:
        nahuatl: Изречението на науатл
        candidates: Списък от 10 испански кандидат-изречения
    
    Връща:
        Индекс (0-9) на предсказания правилен превод
    """
    # Кодиране на заявката и кандидатите
    query_embedding = model.encode(nahuatl, convert_to_tensor=True)
    candidate_embeddings = model.encode(candidates, convert_to_tensor=True)
    
    # Изчисляване на косинусово сходство
    similarities = util.cos_sim(query_embedding, candidate_embeddings)[0]
    
    # Връщане на индекса с най-високо сходство
    return torch.argmax(similarities).item()

# ============================================================================
# ВАШЕТО РЕШЕНИЕ - КРАЙ
# ============================================================================

---

## Оценка и предаване (не редактирайте по-долу)

Кодът по-долу:
1. Изпълнява вашата функция `predict` върху валидационния набор и показва вашия резултат
2. Изпълнява предсказания върху тестовия набор и генерира `submission.csv`

In [9]:
def run_predictions(df: pd.DataFrame) -> List[int]:
    """Изпълнява predict() върху всички заявки в dataframe."""
    predictions = []
    for idx, row in df.iterrows():
        nahuatl = row['nahuatl']
        candidates = [row[f'spanish_{i}'] for i in range(10)]
        pred = predict(nahuatl, candidates)
        assert pred in list(range(0,10)), f"Вашето предсказание {pred} не е цяло число между 0 и 9" 
        predictions.append(pred)
        if (idx + 1) % 100 == 0:
            print(f"  Обработени {idx + 1}/{len(df)} заявки...")
    return predictions

# Оценка върху валидационния набор
print("Оценяване върху валидационния набор...")
val_predictions = run_predictions(val_df)

correct = sum(1 for pred, gt in zip(val_predictions, val_ground_truth) if pred == gt)
accuracy = correct / len(val_ground_truth)

print("\n" + "="*50)
print(f"ВАЛИДАЦИОНЕН РЕЗУЛТАТ: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Правилни: {correct}/{len(val_ground_truth)}")
print("="*50)

Оценяване върху валидационния набор...
  Обработени 100/500 заявки...
  Обработени 200/500 заявки...
  Обработени 300/500 заявки...
  Обработени 400/500 заявки...
  Обработени 500/500 заявки...

ВАЛИДАЦИОНЕН РЕЗУЛТАТ: 0.3360 (33.60%)
Правилни: 168/500


In [ ]:
USER_ID = ''
assert USER_ID != '', "Моля въведете своето ID"
# Генериране на тестови предсказания
print("\nГенериране на тестови предсказания...")
test_predictions = run_predictions(test_df)

# Запазване в submission.csv
submission_filename = f'Task_2_{USER_ID}_submission.csv'
pd.DataFrame(test_predictions).to_csv(submission_filename, index=False, header=False)

print("\n" + "="*50)
print("ГОТОВ ЗА ПРЕДАВАНЕ")
print("="*50)
print(f"  {submission_filename}.csv: {len(test_predictions)} предсказания")
print(f"  {submission_filename}.csv: готов за качване")
print("="*50)